In [ ]:
import pandas as pd
import numpy as np 

df_vendas = pd.read_csv('vendas_tech.csv', low_memory=False)
df_gerentes = pd.read_excel('gerentes_lojas.xlsx')

display(df_vendas)

display(df_vendas.columns)
display(df_vendas.info())




In [ ]:
#Tratamento de dados

df_tratadov = df_vendas.copy()

#COLUNAS#
df_tratadov = df_tratadov.drop(columns=['Data_Base'])

#NULOS#
#df_tratadov = df_tratadov.dropna() exclui todas as linhas q tenham valor vazio
df_tratadov['Loja'] = df_tratadov['Loja'].fillna('Online') #substitui os valores nulos da coluna Loja por Online

#TIPO DE DADOS#
df_tratadov['Data'] = pd.to_datetime(df_tratadov['Data'], format='%Y-%m-%d') #converte a coluna Data para o tipo datetime


#PADRONIZAÇÃO DE DADOS#
df_tratadov['Loja'] = df_tratadov['Loja'].str.strip() #remove espaços em branco no inicio e no final dos valores da coluna Loja
df_tratadov['Loja'] = df_tratadov['Loja'].str.title() #converte todos os valores da coluna Loja para maiusculo inicial

#encadeamento
#df_tratadov['Loja'] = df_tratadov['Loja'].str.strip().str.title() #remove espaços em branco no inicio e no final dos valores da coluna Loja e converte todos os valores da coluna Loja para maiusculo inicial

#DUPLICATAS#
df_tratadov = df_tratadov.drop_duplicates(subset=['ID_Pedido']) #exclui todas as linhas duplicadas do dataframe



display(df_tratadov)
display(df_tratadov.info())


In [ ]:
#Criar novas colunas#

# faturmento
df_tratadov['Faturamento'] = df_tratadov['Qtd'] * df_tratadov['Preco_Unitario']

# forma de venda (o np.where() é uma função do numpy que permite criar uma nova coluna com base em uma condição. A sintaxe é: np.where(condição, valor_se_verdadeiro, valor_se_falso))
df_tratadov['Forma_Venda'] = np.where(df_tratadov['Loja'] == 'Online', 'Online', 'Presencial')


# regiao
display(df_tratadov['Loja'].unique())

dict_regioes = {
    'São Paulo': 'Sudeste',
    'Belo Horizonte': 'Sudeste',
    'Online': 'Online',
    'Rio De Janeiro': 'Sudeste',
    'Salvador': 'Nordeste',
    'Recife': 'Nordeste',
    'Curitiba': 'Sul',
    'Porto Alegre': 'Sul'
}
df_tratadov['Regiao'] = df_tratadov['Loja'].map(dict_regioes)



display(df_tratadov)
display(df_tratadov.info())
display(df_tratadov.isna().sum()) #verifica se existem valores nulos no dataframe

In [ ]:
#Analisar -> filtrar
df_tratadov = df_tratadov.sort_values(by='Data')#pode ser adicionado mais uma coluna usando lista e para mudar a ordem voce coloca o ascending = false #ordena o dataframe pela coluna Data
df_tratadov = df_tratadov.reset_index(drop=True) #reseta o index do dataframe sem criar uma nova coluna com o index antigo por conta do drop=True


#.loc: id do pedido -> por nome da coluna e por nome da linha
id_pedido = 4

loja = df_tratadov.loc[3, 'Loja'] #retorna o valor da coluna Loja da linha 3
produto = df_tratadov.loc[3, 'Produto'] #retorna o valor da coluna Produto da linha 3
cliente = df_tratadov.loc[3, 'Cliente'] #retorna o valor da coluna Cliente da linha 3
print(loja, produto, cliente)

#.iloc -> por posição
id_pedido = 4
loja = df_tratadov.iloc[3, 2] 
produto = df_tratadov.iloc[3, 3] 
cliente = df_tratadov.iloc[3, 6] 

print(loja, produto, cliente)


#condicional
df_id_pedido = df_tratadov[df_tratadov['ID_Pedido'] == 4] #retorna todas as linhas do dataframe onde a coluna ID_Pedido é igual a 4



#exportar pedaços da base
df_vendas_sp = df_tratadov[df_tratadov['Loja'] == 'São Paulo'] #retorna todas as linhas do dataframe onde a coluna Loja é igual a São Paulo
df_vendas_sp.to_csv('vendas_sp.csv', index=False) #exporta o dataframe para um arquivo csv sem o index (index=False)

#exportar as vendas de 2024
df_vendas_2024 = df_tratadov[df_tratadov['Data'].dt.year == 2024] #retorna todas as linhas do dataframe onde a coluna Data é igual a 2024
                                               #pode se usado tambem o >= '2024-01-01' 

#duplas condições

df_vendas_HDMI_SUL = df_tratadov[(df_tratadov['Produto']=='Cabo HDMI') & (df_tratadov['Regiao']=='Sul')] #retorna todas as linhas do dataframe onde a coluna Produto é igual a Cabo HDMI e a coluna Regiao é igual a Sul



display(df_tratadov)
display(df_id_pedido)
display(df_vendas_sp)
display(df_vendas_2024)
display(df_vendas_HDMI_SUL)


In [ ]:
#analises por agrupamento
display(df_tratadov)

#analise_lojas = df_tratadov.groupby('Loja').sum() #forma que retorna meio errado
analise_lojas = df_tratadov[['Loja', 'Faturamento']].groupby('Loja').sum() #forma correta de agrupar por Loja e somar o faturamento
analise_lojas = analise_lojas.sort_values(by='Faturamento', ascending=False) #ordena o dataframe pela coluna Faturamento em ordem decrescente
analise_lojas = analise_lojas.reset_index()
#TOMAR CUIDADO com essa parte de ordenar o dataframe, pois ao fazer isso os valores se tornam str e nao é possivel fazer operações matemáticas com eles, então é importante fazer a ordenação depois de fazer as operações matemáticas
analise_lojas['Faturamento'] = analise_lojas['Faturamento'].map('R${:,.2f}'.format) #formata a coluna Faturamento para o formato de moeda brasileira


#ranking de produtos que mais venderam no Online
df_vendas_online = df_tratadov[df_tratadov['Loja'] == 'Online'] #retorna todas as linhas do dataframe onde a coluna Loja é igual a Online
analise_produtos_online = df_vendas_online[['Produto', 'Qtd']].groupby('Produto').sum() #agrupa por Produto e soma o Faturamento
analise_produtos_online = analise_produtos_online.sort_values(by='Qtd', ascending=False) #ordena o dataframe pela coluna Faturamento em ordem decrescente
#mudar o nome da coluna para apresentação
analise_produtos_online = analise_produtos_online.rename(columns={'Qtd': 'Vendas totais'}) #renomeia a coluna Qtd para Vendas totais
display(analise_produtos_online)

 
#analise de ranking por loja e por produto
#quais produtos mais venderam em cada loja
analise_produtos_em_lojas = df_tratadov[['Loja', 'Produto', 'Qtd']].groupby(['Loja', 'Produto']).sum() #agrupa por Loja e Produto e soma o Faturamento

#with pd.option_context('display.max_rows', None):#exibe todas as linhas do dataframe

#display(analise_produtos_em_lojas)


display(df_vendas_online)
display(analise_lojas)

